# Reto 1: Humano vs Máquina - Clasificación de Pingüinos Palmer

**Dataset:** Palmer Penguins  
**Objetivo:** Comparar un clasificador basado en reglas manuales contra un modelo de Machine Learning

---

## Configuración Inicial

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
np.random.seed(42)

COLORES = {'Adelie': '#FF6B6B', 'Chinstrap': '#4ECDC4', 'Gentoo': '#45B7D1'}

In [ ]:
df_original = sns.load_dataset('penguins')
df = df_original.dropna().reset_index(drop=True)

print(f"Registros totales: {len(df)}")
print(f"Islas: {df['island'].nunique()}")
print(f"Especies: {df['species'].nunique()}")
print()
print(df['species'].value_counts())

---

## Parte 1: Exploración del Dataset (15 puntos)

### Ejercicio 1.1: Estadísticas Básicas

In [ ]:
df.describe()

**Respuestas:**

1. Rango de masa corporal: de **2700 g** a **6300 g**
2. Longitud promedio del pico: **43.99 mm**
3. Longitud promedio de la aleta: **200.97 mm**

### Ejercicio 1.2: Estadísticas por Especie

In [ ]:
columnas_numericas = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']

promedios = df.groupby('species')[columnas_numericas].mean().round(2)
print(promedios)

**Respuestas:**

1. Especie con pico más largo: **Chinstrap** (~48.83 mm)
2. Especie con aletas más largas: **Gentoo** (~217.19 mm)
3. Especie más pesada: **Gentoo** (~5076 g)
4. Especie con mayor profundidad de pico: **Adelie** (~18.35 mm)

### Visualizaciones

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

variables = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
titulos = ['Longitud del Pico (mm)', 'Profundidad del Pico (mm)',
           'Longitud de Aleta (mm)', 'Masa Corporal (g)']

for ax, var, titulo in zip(axes.flat, variables, titulos):
    for species in df['species'].unique():
        subset = df[df['species'] == species]
        ax.hist(subset[var], alpha=0.6, label=species,
                color=COLORES[species], bins=20, edgecolor='white')
    ax.set_xlabel(titulo)
    ax.set_ylabel('Frecuencia')
    ax.legend()
    ax.set_title(f'Distribución de {titulo} por Especie')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for species in df['species'].unique():
    subset = df[df['species'] == species]
    axes[0].scatter(subset['bill_length_mm'], subset['bill_depth_mm'],
                   c=COLORES[species], label=species, alpha=0.7, s=60)
axes[0].set_xlabel('Longitud del Pico (mm)')
axes[0].set_ylabel('Profundidad del Pico (mm)')
axes[0].set_title('Dimensiones del Pico por Especie')
axes[0].legend()

for species in df['species'].unique():
    subset = df[df['species'] == species]
    axes[1].scatter(subset['flipper_length_mm'], subset['body_mass_g'],
                   c=COLORES[species], label=species, alpha=0.7, s=60)
axes[1].set_xlabel('Longitud de Aleta (mm)')
axes[1].set_ylabel('Masa Corporal (g)')
axes[1].set_title('Aleta vs Masa por Especie')
axes[1].legend()

plt.tight_layout()
plt.show()

---

## Parte 2: Diseño de Reglas de Clasificación (25 puntos)

### Ejercicio 2.1: Estrategia

1. **Variable principal:** `flipper_length_mm`. Gentoo tiene aletas notablemente más largas (203-231 mm) comparado con Adelie (172-210) y Chinstrap (178-212). Un umbral alrededor de 207 mm separa Gentoo casi perfectamente.

2. **Umbrales planificados:**
   - `flipper_length >= 207` → Gentoo
   - `bill_depth <= 15.5` → Gentoo (pico muy delgado, característica de Gentoo)
   - `bill_length >= 46` → Chinstrap (pico largo)
   - `bill_length < 37` → Adelie (Chinstrap mínimo es ~41 mm)
   - Zona gris (41-46 mm): usar `bill_depth >= 18.5` → Adelie

3. **Especie más fácil:** Gentoo, porque sus aletas son significativamente más largas y su pico mucho más delgado. Hay poca superposición con las otras dos especies.

4. **Especie más difícil:** Adelie vs Chinstrap. Ambas tienen aletas similares y sus rangos de longitud de pico se solapan parcialmente en el rango 41-46 mm. Requiere combinar múltiples variables.

### Ejercicio 2.2: Implementación del Clasificador Humano

In [ ]:
def clasificador_humano(bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g):
    """
    Clasifica un pinguino basandose en reglas diseñadas manualmente.

    Estrategia:
    1. Gentoo: aleta >= 207mm, o pico delgado (bd < 15.9) combinado con aleta >= 203mm.
       El doble criterio evita capturar Adelie con bill_depth bajo y aleta corta.
    2. Chinstrap con bill_depth >= 20 siempre tiene bill_length >= 50; Adelie no.
    3. bill_length < 37 es siempre Adelie (Chinstrap minimo es ~41mm).
    4. bill_length >= 46 es siempre Chinstrap.
    5. Zona 37-41: Adelie si bill_depth >= 16.5.
    6. Zona gris 41-46 con bd >= 18.5: Adelie salvo que bl >= 45 (Chinstrap largo).
    7. Zona gris 41-46 con bd 17-18.5: usar body_mass >= 3900 como desempate.
    """
    if flipper_length_mm >= 207:
        return 'Gentoo'

    if bill_depth_mm < 15.9 and flipper_length_mm >= 203:
        return 'Gentoo'

    if bill_depth_mm >= 20.0:
        if bill_length_mm >= 50:
            return 'Chinstrap'
        return 'Adelie'

    if bill_length_mm < 37:
        return 'Adelie'

    if bill_length_mm >= 46:
        return 'Chinstrap'

    if bill_length_mm <= 41:
        if bill_depth_mm >= 16.5:
            return 'Adelie'
        return 'Chinstrap'

    if bill_depth_mm >= 18.5:
        if bill_length_mm >= 45.0:
            return 'Chinstrap'
        return 'Adelie'

    if bill_depth_mm >= 17.0:
        if body_mass_g >= 3900:
            return 'Adelie'
        return 'Chinstrap'

    return 'Chinstrap'

In [ ]:
casos_prueba = [
    [39.1, 18.7, 181, 3750, 'Adelie'],
    [46.5, 17.9, 192, 3500, 'Chinstrap'],
    [46.1, 13.2, 211, 4500, 'Gentoo'],
]

print(f"{'Pico L':>8} {'Pico D':>8} {'Aleta':>8} {'Masa':>8} | {'Real':>12} {'Pred':>12} {'OK?':>6}")
print("-" * 70)

for caso in casos_prueba:
    pred = clasificador_humano(caso[0], caso[1], caso[2], caso[3])
    correcto = "SI" if pred == caso[4] else "NO"
    print(f"{caso[0]:>8.1f} {caso[1]:>8.1f} {caso[2]:>8} {caso[3]:>8} | {caso[4]:>12} {pred:>12} {correcto:>6}")

---

## Parte 3: Evaluación del Clasificador Humano (20 puntos)

### Ejercicio 3.1: División de Datos

In [ ]:
X = df[['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']]
y = df['species']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Entrenamiento: {len(X_train)} pingüinos")
print(f"Prueba: {len(X_test)} pingüinos")

### Ejercicio 3.2: Evaluación

In [ ]:
predicciones_humano = []

for idx, row in X_test.iterrows():
    pred = clasificador_humano(
        row['bill_length_mm'],
        row['bill_depth_mm'],
        row['flipper_length_mm'],
        row['body_mass_g']
    )
    predicciones_humano.append(pred)

accuracy_humano = accuracy_score(y_test, predicciones_humano)

print(f"Accuracy del clasificador humano: {accuracy_humano:.2%}")
print(f"Aciertos: {int(accuracy_humano * len(y_test))} de {len(y_test)} pingüinos")

In [ ]:
cm_humano = confusion_matrix(y_test, predicciones_humano, labels=['Adelie', 'Chinstrap', 'Gentoo'])

plt.figure(figsize=(8, 6))
sns.heatmap(cm_humano, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Adelie', 'Chinstrap', 'Gentoo'],
            yticklabels=['Adelie', 'Chinstrap', 'Gentoo'])
plt.xlabel('Predicción del Humano')
plt.ylabel('Especie Real')
plt.title('Matriz de Confusión - Clasificador Humano')
plt.tight_layout()
plt.show()

print(classification_report(y_test, predicciones_humano))

**Reflexión:**

1. La especie mejor clasificada es **Gentoo**, por la separación clara en la longitud de aleta.
2. La mayor dificultad estuvo en la frontera **Adelie-Chinstrap** en el rango de bill_length 41-46 mm, donde ambas especies se solapan.
3. Los errores corresponden principalmente a individuos en ese rango límite donde ninguna regla simple es perfecta.

---

## Parte 4: El Clasificador de Machine Learning (20 puntos)

### Ejercicio 4.1: Entrenamiento

In [ ]:
modelo_ml = DecisionTreeClassifier(random_state=42)
modelo_ml.fit(X_train, y_train)

print(f"Modelo entrenado")
print(f"Profundidad del árbol: {modelo_ml.get_depth()}")
print(f"Número de hojas: {modelo_ml.get_n_leaves()}")

### Ejercicio 4.2: Evaluación del Modelo ML

In [ ]:
predicciones_ml = modelo_ml.predict(X_test)
accuracy_ml = accuracy_score(y_test, predicciones_ml)

print(f"Accuracy del modelo ML: {accuracy_ml:.2%}")
print(f"Aciertos: {int(accuracy_ml * len(y_test))} de {len(y_test)} pingüinos")

In [ ]:
cm_ml = confusion_matrix(y_test, predicciones_ml, labels=['Adelie', 'Chinstrap', 'Gentoo'])

plt.figure(figsize=(8, 6))
sns.heatmap(cm_ml, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Adelie', 'Chinstrap', 'Gentoo'],
            yticklabels=['Adelie', 'Chinstrap', 'Gentoo'])
plt.xlabel('Predicción ML')
plt.ylabel('Especie Real')
plt.title('Matriz de Confusión - Modelo ML')
plt.tight_layout()
plt.show()

print(classification_report(y_test, predicciones_ml))

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(20, 10))
plot_tree(
    modelo_ml,
    feature_names=['bill_length', 'bill_depth', 'flipper_length', 'body_mass'],
    class_names=['Adelie', 'Chinstrap', 'Gentoo'],
    filled=True, rounded=True, fontsize=9
)
plt.title('Árbol de Decisión Aprendido por la Máquina', fontsize=15)
plt.tight_layout()
plt.show()

---

## Parte 5: La Batalla Final (20 puntos)

In [ ]:
diferencia = accuracy_ml - accuracy_humano

print(f"Clasificador Humano:  {accuracy_humano:.2%} ({int(accuracy_humano * len(y_test))}/{len(y_test)})")
print(f"Clasificador ML:      {accuracy_ml:.2%} ({int(accuracy_ml * len(y_test))}/{len(y_test)})")
print()

if diferencia > 0:
    print(f"Ganador: LA MAQUINA (+{diferencia:.2%})")
elif diferencia < 0:
    print(f"Ganador: EL HUMANO (+{-diferencia:.2%})")
else:
    print("EMPATE")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

clasificadores = ['Humano', 'Máquina (ML)']
accuracies = [accuracy_humano * 100, accuracy_ml * 100]
colores_barras = ['#FF6B6B', '#4ECDC4']

bars = axes[0].bar(clasificadores, accuracies, color=colores_barras, edgecolor='white', linewidth=2)
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Comparación de Accuracy', fontweight='bold')
axes[0].set_ylim(0, 105)
for bar, acc in zip(bars, accuracies):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{acc:.1f}%', ha='center', va='bottom', fontsize=14, fontweight='bold')

sns.heatmap(cm_humano, annot=True, fmt='d', cmap='Oranges', ax=axes[1],
            xticklabels=['A', 'C', 'G'], yticklabels=['A', 'C', 'G'])
axes[1].set_title('Matriz Humano', fontweight='bold')
axes[1].set_xlabel('Predicción')
axes[1].set_ylabel('Real')

sns.heatmap(cm_ml, annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=['A', 'C', 'G'], yticklabels=['A', 'C', 'G'])
axes[2].set_title('Matriz ML', fontweight='bold')
axes[2].set_xlabel('Predicción')
axes[2].set_ylabel('Real')

plt.suptitle('Batalla Final: Humano vs Máquina', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### Ejercicio 5.1: Análisis de Errores

In [ ]:
resultados = X_test.copy()
resultados['especie_real'] = y_test.values
resultados['pred_humano'] = predicciones_humano
resultados['pred_ml'] = predicciones_ml
resultados['error_humano'] = resultados['especie_real'] != resultados['pred_humano']
resultados['error_ml'] = resultados['especie_real'] != resultados['pred_ml']

errores_humano = resultados[resultados['error_humano']]
print(f"Errores del clasificador HUMANO ({len(errores_humano)} casos):")
if len(errores_humano) > 0:
    print(errores_humano[['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm',
                           'body_mass_g', 'especie_real', 'pred_humano']].to_string())
else:
    print("Ningun error")

In [ ]:
errores_ml = resultados[resultados['error_ml']]
print(f"Errores del clasificador ML ({len(errores_ml)} casos):")
if len(errores_ml) > 0:
    print(errores_ml[['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm',
                       'body_mass_g', 'especie_real', 'pred_ml']].to_string())
else:
    print("Ningun error")

### Ejercicio 5.2: Reflexión Final

1. **¿Quién ganó?** El modelo ML ganó por un pequeño margen. Encontró thresholds óptimos mediante optimización matemática sobre todos los datos, mientras que el humano los estimó visualmente.

2. **Errores del humano vs máquina:** El humano cometió más errores en la zona límite Adelie-Chinstrap (bill_length 41-46 mm). El árbol de decisión encontró thresholds más precisos (ej. 43.35 vs el 41 que use yo) que reducen esa ambigüedad.

3. **¿Las reglas son similares?** Sí, la primera división del árbol usa `flipper_length <= 206.5`, que coincide con mi umbral de 207. Ambos identifican Gentoo por las aletas primero. La diferencia está en los thresholds exactos para Adelie/Chinstrap.

4. **Principal ventaja del ML:** El modelo encuentra los thresholds óptimos matemáticamente, sin sesgo humano. También puede combinar múltiples variables simultáneamente de forma más eficiente que reglas manuales.

5. **¿Qué haría diferente?** Añadiría más condiciones en la zona gris 41-46 mm usando `body_mass_g` como variable adicional de desempate, ya que Adelie tiende a ser más ligero que Chinstrap.

---

## Bonus: Mejora del Clasificador (+10 puntos)

In [ ]:
def clasificador_humano_v2(bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g):
    """
    Version mejorada tras analizar los errores de la v1.

    Cambios:
    1. Se ajusto el umbral de la zona gris de 41 a 43.35 (inspirado en el arbol ML)
    2. Se agrego body_mass como criterio de desempate en la zona ambigua
    3. Se refino el limite inferior de bill_length para Gentoo residual
    """
    if flipper_length_mm >= 207:
        return 'Gentoo'

    if bill_depth_mm <= 15.5:
        return 'Gentoo'

    if bill_depth_mm >= 20.0:
        return 'Adelie'

    if bill_length_mm < 37:
        return 'Adelie'

    if bill_length_mm >= 46:
        if bill_depth_mm >= 20.0:
            return 'Adelie'
        return 'Chinstrap'

    # Umbral mejorado: 43.35 en lugar de 41 (del arbol ML)
    if bill_length_mm <= 43.35:
        if bill_depth_mm <= 15.95:
            return 'Chinstrap'
        return 'Adelie'

    # Zona 43.35 - 46: usar bill_depth y body_mass
    if bill_depth_mm >= 18.5:
        return 'Adelie'
    if bill_depth_mm <= 17.65:
        return 'Chinstrap'
    # empate final: masa corporal (Adelie ligeramente mas ligero)
    if body_mass_g <= 3600:
        return 'Adelie'
    return 'Chinstrap'

In [ ]:
predicciones_humano_v2 = []

for idx, row in X_test.iterrows():
    pred = clasificador_humano_v2(
        row['bill_length_mm'],
        row['bill_depth_mm'],
        row['flipper_length_mm'],
        row['body_mass_g']
    )
    predicciones_humano_v2.append(pred)

accuracy_humano_v2 = accuracy_score(y_test, predicciones_humano_v2)

print(f"Version 1: {accuracy_humano:.2%}")
print(f"Version 2: {accuracy_humano_v2:.2%}")
print(f"Mejora:    {(accuracy_humano_v2 - accuracy_humano):.2%}")

if accuracy_humano_v2 > accuracy_humano:
    print("El clasificador mejoro.")
else:
    print("Sin mejora.")

**Explicación de cambios:**

- El árbol de decisión usa `bill_length <= 43.35` como primer split para Adelie/Chinstrap. Adopté ese umbral en lugar del 41 original, lo que reduce errores en la zona de solapamiento.
- Añadí `body_mass_g` como criterio de desempate final: cuando bill_depth y bill_length son ambiguos, una masa menor tiende a indicar Adelie.
- Separé el caso `bill_length >= 46` de forma más explícita para evitar falsos Chinstrap con bill_depth muy alto.